# Foundry web-search blocklist with regex URL redaction

APIM merges the organization blocklist into hosted search requests. JSON requests go directly from APIM to Foundry, including when web search runs. APIM preserves the body and sets `x-response-metrics`, `x-response-metrics-status`, and `x-response-request-id`. For SSE, a live `/api/responses` relay calls the separate regex Function once per text-delta pair or non-text event after an actual search invocation appears. The redactor replaces each complete matching URL with `[BLOCKED LINK]`, with no second model request.

**Audience:** Developers using APIM, Azure Functions, and Foundry Responses.
**Prerequisites:** Python 3.12+, Azure CLI with `az login`, a Foundry deployment supporting Responses and `web_search`, Contributor plus RBAC Administrator (or Owner), and permission to create Entra app registrations. Run `uv sync --group dev` from this directory, select its `.venv` kernel, and copy `.env.example` to `.env`.

You will validate locally, deploy, and test JSON and SSE with no tool offered, an unused tool, and actual search. SSE joins consecutive text deltas in pairs without collecting the full response. URLs split within a pair can be replaced; URLs split across separate pairs can still remain. An unused tool skips the regex route.

Deployment creates billable resources: a B1 Function App, private host storage/networking, and three secret-free registrations. APIM is reused when configured, otherwise created. Foundry is reused and receives one model request per client call. See [README.md](README.md) and the separate cleanup notebook.

## 1. Check regex redaction locally
These examples run without Azure credentials. Each complete blocked URL becomes `[BLOCKED LINK]`, including its path, query, and fragment. Markdown markup, link labels, surrounding punctuation, and spacing stay unchanged.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

LAB = Path.cwd()
assert LAB.name == 'ai-foundry-web-search-blocklist-function', 'Use this lab as the working directory.'
sys.path.insert(0, str(LAB / 'src'))
from filtering import DomainRedactor, domain_list
from processing import invoked_web_search

required_domains = domain_list(json.loads((LAB / 'blocked-domains.json').read_text()))
redactor = DomainRedactor(required_domains)
original = 'Read  [video](https://youtube.com/watch?v=demo) and [docs](https://learn.microsoft.com/azure/).\n'
expected = 'Read  [video]([BLOCKED LINK]) and [docs](https://learn.microsoft.com/azure/).\n'
assert redactor.text(original) == expected
assert redactor.text(expected) == expected
print(repr(expected))
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], check=True)


### Exercise: hostname boundaries and actual tool use
Add `example.com` to the blocklist. Its subdomains match, while the same text in
an allowed URL's query does not. Merely declaring `web_search` does not count as an invocation.


In [ ]:
extra_domains = domain_list([*required_domains, 'example.com'])
example = 'A https://news.example.com/item B https://learn.microsoft.com/?q=example.com'
assert DomainRedactor(extra_domains).text(example) == 'A [BLOCKED LINK] B https://learn.microsoft.com/?q=example.com'
assert not invoked_web_search({'tools': [{'type': 'web_search'}], 'output': []})
assert invoked_web_search({'output': [{'type': 'web_search_call', 'status': 'completed'}]})
print('Regex hostname matching and invocation detection passed.')


from streaming import redact_events
first = {"type": "response.output_text.delta", "item_id": "msg_pair", "output_index": 1,
         "content_index": 0, "delta": "Before https://you"}
second = {**first, "delta": "tube.com/a after"}
events = redact_events({"events": [first, second],
                        "search_event": {"type": "response.web_search_call.searching"}},
                       DomainRedactor(["youtube.com"]))
assert [event["delta"] for event in events] == ["Before [BLOCKED LINK]", " after"]
print("Two text deltas filtered together:", "".join(event["delta"] for event in events))


## 2. Configure Azure and the existing deployment
Use a separate resource group for this lab. The organization domains in
`blocked-domains.json` configure both APIM's search filter and the regex Function.
The notebook validates them before deploying. Tokens stay in memory.


In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlsplit
from uuid import UUID

import httpx
import msal
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from lab_helpers import az, build_function_package, create_lab_apps, iter_sse, output_text, wait_for_function_indexing

LAB = Path.cwd()
assert (LAB / 'main.bicep').exists(), 'Use labs/ai-foundry-web-search-blocklist-function as the working directory.'
load_dotenv(LAB / '.env')
sys.path.insert(0, str(LAB / 'src'))
from filtering import DomainRedactor, domain_list
from processing import invoked_web_search
required_domains = domain_list(json.loads((LAB / "blocked-domains.json").read_text()))

resource_group = os.getenv('LAB_RESOURCE_GROUP', 'lab-ai-foundry-web-search-blocklist-function')
location = os.getenv('LAB_LOCATION', 'westus2')
apim_sku = os.getenv('LAB_APIM_SKU', 'Basicv2')
foundry_name = os.environ['FOUNDRY_NAME']
foundry_group = os.environ['FOUNDRY_RESOURCE_GROUP']
foundry_deployment = os.getenv('FOUNDRY_DEPLOYMENT', 'gpt-5.6-luna')
account = az('account', 'show')
tenant_id, subscription_id = account['tenantId'], account['id']
foundry_subscription = os.getenv('FOUNDRY_SUBSCRIPTION_ID', subscription_id)
user = az('ad', 'signed-in-user', 'show')
caller_object_id = str(UUID(user['id']))
publisher_email = os.getenv('LAB_PUBLISHER_EMAIL') or user.get('mail') or user['userPrincipalName']

foundry = az('cognitiveservices', 'account', 'show', '--name', foundry_name,
             '--resource-group', foundry_group, '--subscription', foundry_subscription)
subdomain = foundry['properties']['customSubDomainName']
foundry_endpoint = os.getenv('FOUNDRY_ENDPOINT', f'https://{subdomain}.openai.azure.com/openai/v1').rstrip('/')
parsed = urlsplit(foundry_endpoint)
assert parsed.scheme == 'https' and parsed.path == '/openai/v1'
assert not parsed.query and not parsed.fragment and not parsed.username and not parsed.password
assert parsed.port in (None, 443) and parsed.hostname.endswith(('.openai.azure.com', '.services.ai.azure.com'))
assert re.fullmatch(r'[A-Za-z0-9_.-]+', foundry_deployment), 'Deployment name contains unsupported policy characters.'
assert resource_group != foundry_group or subscription_id != foundry_subscription, 'Use a separate lab resource group for safe cleanup.'
models = az('cognitiveservices', 'account', 'deployment', 'list', '--name', foundry_name,
            '--resource-group', foundry_group, '--subscription', foundry_subscription)
selected = next((model for model in models if model['name'] == foundry_deployment), None)
assert selected, f'Create the {foundry_deployment!r} Foundry deployment before continuing.'
assert selected['properties']['provisioningState'] == 'Succeeded', 'The configured model deployment is not ready.'
print(f'Tenant: {tenant_id}; subscription: {subscription_id}; lab resource group: {resource_group}')
print(f'Foundry deployment: {foundry_deployment}; region: {foundry["location"]}')

lab_tag = 'ai-foundry-web-search-blocklist-function'
if az('group', 'exists', '--name', resource_group):
    existing_group = az('group', 'show', '--name', resource_group)
    assert (existing_group.get('tags') or {}).get('ai-gateway-lab') == lab_tag, 'Choose a new resource group; this group is not owned by this lab.'

# APIM_SERVICE_ID reuses the gateway configured in the source blocklist lab.
existing_apim_id = os.getenv('APIM_SERVICE_ID', '')
existing_apim_params = {}
existing_foundry_assignments = []
if existing_apim_id:
    apim_parts = existing_apim_id.strip('/').split('/')
    assert len(apim_parts) == 8 and apim_parts[6].lower() == 'service'
    existing_apim = az('apim', 'show', '--name', apim_parts[7], '--resource-group', apim_parts[3], '--subscription', apim_parts[1])
    assert (existing_apim.get('identity') or {}).get('principalId'), 'Enable the existing APIM system-assigned identity first.'
    existing_apim_params = {
        'existingApimName': apim_parts[7], 'existingApimResourceGroup': apim_parts[3],
        'existingApimSubscriptionId': apim_parts[1],
    }
    assignments = az('role', 'assignment', 'list', '--scope', foundry['id'], '--subscription', foundry_subscription, '--include-inherited')
    existing_foundry_assignments = [item['id'] for item in assignments
        if item['principalId'] == existing_apim['identity']['principalId']
        and item['roleDefinitionId'].endswith('/5e0bd9bd-7b93-4f28-af87-19fc36ad61bd')]
    print('Reusing APIM:', existing_apim['name'])


## 3. Create Entra registrations
Create Gateway API, notebook public client, and Function API registrations with
no passwords or certificates. Easy Auth permits APIM and the Function App's managed identity, so the relay can invoke the regex route.
The Function's delegated test scope lets the notebook prove that even a valid
user token cannot call the Function directly.


In [ ]:
state_path = LAB / '.lab-state.json'
state = json.loads(state_path.read_text()) if state_path.exists() else {'tenant_id': tenant_id}
assert state['tenant_id'] == tenant_id
if state.get('subscription_id'):
    assert state['subscription_id'] == subscription_id and state['resource_group'] == resource_group
state.update(subscription_id=subscription_id, resource_group=resource_group,
             foundry_subscription_id=foundry_subscription, foundry_id=foundry['id'],
             deployment_name='web-search-blocklist-function', lab_name=lab_tag,
             apim_reused=bool(existing_apim_id), apim_service_id=existing_apim_id,
             existing_foundry_role_assignment_ids=existing_foundry_assignments)
# Persist cleanup coordinates before creating the first registration.
state_path.write_text(json.dumps(state, indent=2) + '\n')
state = create_lab_apps(state_path, tenant_id, resource_group)
gateway_client_id = state['gateway_app']['appId']
function_client_id = state['function_app']['appId']
print('Three Entra applications are ready; no secrets were created.')


## 4. Deploy infrastructure and publish the Functions

Set `APIM_SERVICE_ID` to reuse an existing gateway; otherwise APIM creation can take 30–60 minutes. APIM receives or reuses Cognitive Services OpenAI User on Foundry. The Function App identity receives its own Foundry role for the SSE relay and a host-storage role. Easy Auth permits APIM and the Function identity; user tokens remain rejected.

The Function App exposes `/api/responses` for live SSE and `/api/redact` for paired text-delta and single-event redaction. The handler retains its legacy JSON envelope API, but the gateway never sends JSON responses to it. Both routes share the same B1 plan. The regex handler itself performs no model or network calls. The relay calls it only after an observed search invocation.

The optional workspace setting enables GatewayLogs without response-body logging. Changing the organization blocklist requires redeployment. Publish both routes and apply the APIM routing update together.

In [ ]:
az('group', 'create', '--name', resource_group, '--location', location, '--tags', f'ai-gateway-lab={lab_tag}')
params = {
    'location': location, 'apimSku': apim_sku, 'publisherEmail': publisher_email,
    'foundryName': foundry_name, 'foundryResourceGroup': foundry_group,
    'foundrySubscriptionId': foundry_subscription, 'foundryEndpoint': foundry_endpoint,
    'foundryDeployment': foundry_deployment, 'gatewayClientId': gateway_client_id,
    'functionClientId': function_client_id, 'callerObjectId': caller_object_id,
    'blockedDomains': required_domains,
    'logAnalyticsWorkspaceId': os.getenv('LAB_LOG_ANALYTICS_WORKSPACE_ID', ''),
    'createFoundryRoleAssignment': not bool(existing_foundry_assignments),
    **existing_apim_params,
}
(LAB / 'params.json').write_text(json.dumps({
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0', 'parameters': {key: {'value': value} for key, value in params.items()},
}, indent=2))
print('Deploying Azure resources. This may take 30–60 minutes.')
deployment = az('deployment', 'group', 'create', '--name', 'web-search-blocklist-function',
                '--resource-group', resource_group, '--template-file', 'main.bicep',
                '--parameters', '@params.json')
outputs = {key: item['value'] for key, item in deployment['properties']['outputs'].items()}
state['outputs'] = outputs
state_path.write_text(json.dumps(state, indent=2) + '\n')
gateway_url, function_url = outputs['gatewayUrl'], outputs['functionUrl']
print(gateway_url)


In [ ]:
# Bundle only deployable source and Linux Python 3.12 wheels.
# This also works from Windows and avoids a runtime build on the Function host.
package = build_function_package(LAB, LAB / 'function.zip')
print('Publishing the prepared package with Entra authentication.')
_ = az('functionapp', 'deployment', 'source', 'config-zip', '--resource-group', resource_group,
       '--name', outputs['functionAppName'], '--src', str(package), '--build-remote', 'false', '--timeout', '1200')
# Reload the published package and allow the Python worker to register both routes.
_ = az('functionapp', 'restart', '--resource-group', resource_group, '--name', outputs['functionAppName'])
functions = wait_for_function_indexing(resource_group, outputs['functionAppName'])
print('Functions redact and responses are indexed.')


## 5. Verify platform authentication
Check Easy Auth, storage, and publishing. Live calls below verify APIM's Foundry
access and conditional Function routing after role propagation.


In [ ]:
subscription_prefix = f'/subscriptions/{subscription_id}/resourceGroups/{resource_group}'
site_id = f'{subscription_prefix}/providers/Microsoft.Web/sites/{outputs["functionAppName"]}'
auth = az('rest', '--url', f'https://management.azure.com{site_id}/config/authsettingsV2?api-version=2024-04-01')['properties']
assert auth['platform']['enabled'] and auth['globalValidation']['requireAuthentication']
validation = auth['identityProviders']['azureActiveDirectory']['validation']
assert validation['defaultAuthorizationPolicy']['allowedPrincipals']['identities'] == [outputs['apimPrincipalId'], outputs['functionPrincipalId']]
assert function_client_id in validation['allowedAudiences']
for name in ('scm', 'ftp'):
    publishing = az('rest', '--url', f'https://management.azure.com{site_id}/basicPublishingCredentialsPolicies/{name}?api-version=2024-04-01')
    assert publishing['properties']['allow'] is False
storage = az('storage', 'account', 'show', '--resource-group', resource_group, '--name', outputs['storageName'])
assert storage['allowSharedKeyAccess'] is False
assert outputs['foundryRoleAssignmentIds'] or state.get('existing_foundry_role_assignment_ids'), 'Foundry inference access is missing.'
print('Easy Auth, APIM and relay identities, disabled publishing passwords, and disabled storage keys verified.')


## 6. Authenticate the notebook
Sign in as the same user returned by Azure CLI. MSAL keeps tokens in kernel
memory and refreshes them before requests. If the tenant blocks device-code flow,
use MSAL interactive browser sign-in in your local environment.


In [ ]:
msal_client = msal.PublicClientApplication(state['notebook_app']['appId'],
                                         authority=f'https://login.microsoftonline.com/{tenant_id}')
def token_for(client_id):
    scopes = [f'api://{client_id}/access_as_user']
    accounts = msal_client.get_accounts()
    result = msal_client.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
    if not result or 'access_token' not in result:
        flow = msal_client.initiate_device_flow(scopes=scopes)
        if 'user_code' not in flow:
            raise RuntimeError(flow.get('error_description', 'Could not initiate sign-in'))
        print(flow['message'])
        result = msal_client.acquire_token_by_device_flow(flow)
    if 'access_token' not in result:
        raise RuntimeError(result.get('error_description', 'Sign-in failed'))
    return result['access_token']

def auth_headers():
    return {'Authorization': f'Bearer {token_for(gateway_client_id)}'}

headers = auth_headers()
print('Gateway token acquired and kept in memory.')


## 7. Test authentication and validation
Unauthenticated gateway calls and direct Function calls must fail. APIM rejects
malformed blocklists before the first model request.


In [ ]:
base_request = {'model': foundry_deployment, 'input': 'Say hello.', 'max_output_tokens': 256}
with httpx.Client(timeout=30) as client:
    assert client.post(gateway_url, json=base_request).status_code == 401
    assert client.post(gateway_url, json=base_request, headers={'Authorization': 'Bearer invalid'}).status_code == 401
    function_token = token_for(function_client_id)
    assert client.post(gateway_url, json=base_request, headers={'Authorization': f'Bearer {function_token}'}).status_code == 401
    assert client.post(function_url, json={}, headers={'Authorization': f'Bearer {function_token}'}).status_code == 403
    assert client.post(function_url, json={}, headers={'x-ms-client-principal': 'spoofed'}).status_code == 401
    for invalid in (
        {'input': 'Test', 'tools': [{'type': 'web_search', 'filters': {'blocked_domains': 'example.com'}}]},
        {'input': 'Test', 'stream': 'true'},
        {'input': 'Test', 'background': True},
        {'input': 'Test', 'tools': [{'type': 'web_search_preview'}]},
    ):
        response = client.post(gateway_url, headers=auth_headers(), json=invalid)
        assert response.status_code == 400, response.text
print('Authentication and pre-inference validation passed.')


## 8. Validate direct JSON requests and metric headers

No-tool, unused-tool, and actual-search JSON requests all go directly to Foundry. APIM preserves the response body and adds the same `x-response-metrics` JSON map, availability status, and request ID. These requests bypass both Function routes; response URLs are unchanged.

In [ ]:
search_request = {
    'model': foundry_deployment,
    'input': 'Use web search to explain Azure API Management response buffering for SSE. Cite documentation.',
    'tools': [{'type': 'web_search', 'filters': {'blocked_domains': ['example.com']}}],
    'tool_choice': 'required', 'reasoning': {'effort': 'low'}, 'max_output_tokens': 4096,
}
expected_domains = domain_list([*required_domains, 'example.com'])
expected_redactor = DomainRedactor(expected_domains)
unused_request = {**search_request, 'tool_choice': 'none', 'input': 'Say hello without searching.'}

def assert_redacted(document):
    assert document['status'] == 'completed'
    assert invoked_web_search(document), 'Expected an actual hosted search output item.'
    assert expected_redactor.response(document) == document, 'An unredacted blocked URL remains.'
    for item in document['output']:
        if item['type'] == 'message':
            for part in item['content']:
                if part['type'] == 'output_text':
                    for annotation in part.get('annotations', []):
                        if 'start_index' in annotation:
                            assert 0 <= annotation['start_index'] <= annotation['end_index'] <= len(part['text'])

with httpx.Client(timeout=240) as client:
    for payload, expected_search in ((base_request, False), (unused_request, False), (search_request, True)):
        response = client.post(gateway_url, headers=auth_headers(), json=payload)
        response.raise_for_status()
        assert response.headers['x-lab-route'] == 'foundry'
        assert response.headers['x-lab-filter'] == 'skipped'
        assert response.headers['x-lab-initial-buffered'] == 'false'
        assert 'x-lab-initial-response-id' not in response.headers
        document = response.json()
        assert document['status'] == 'completed'
        assert invoked_web_search(document) == expected_search
        def metric(value):
            return value if type(value) is int and 0 <= value <= 10**18 else None
        expected = {
            'web_search_count': metric(((document.get('tool_usage') or {}).get('web_search') or {}).get('num_requests')),
            'total_tokens': metric((document.get('usage') or {}).get('total_tokens')),
        }
        assert json.loads(response.headers['x-response-metrics']) == expected
        known = sum(value is not None for value in expected.values())
        assert response.headers['x-response-metrics-status'] == ('reported' if known == 2 else 'partial' if known else 'unavailable')
        assert response.headers['x-response-request-id']
        print('foundry', response.headers['x-response-metrics'])
        display(Markdown(output_text(document)))


## 9. Validate live streaming

No-tool SSE goes directly to Foundry. Requests offering search use `/api/responses` with `x-lab-route: stream-function`, `x-lab-initial-buffered: false`, and `x-response-redaction-window: 2`. An unused tool never invokes the regex route.

After search starts, consecutive `response.output_text.delta` events for the same
message and content part are filtered in non-overlapping pairs. The relay holds
one delta until its partner arrives, joins their text for regex matching, then
returns both original SSE events in order. A URL spanning the pair becomes one
`[BLOCKED LINK]`; its placeholder stays in the first event and the second retains
any following text. IDs, sequence numbers, content indexes, and event counts are
preserved. An unmatched delta is filtered on its own before a non-text event,
a different content part, or an upstream comment. Terminal events flush it before
completion. Unused tools and events before search remain unchanged.

For example, `https://you` plus `tube.com/a` in one pair becomes `[BLOCKED LINK]`.
There is no carry-over between pairs: a URL spanning the second event of one pair
and the first of the next can still escape detection. URLs spanning three or more
deltas, or interrupted by other event types, can also be missed. Paths or queries
arriving in a later pair can remain, and a partial blocked hostname can be
replaced before a later pair adds an allowed suffix. Complete `done` and terminal
snapshots are filtered separately and can differ from concatenated deltas.
Citation offsets are corrected within complete text snapshots; standalone
annotation offsets still refer to the original text.

The relay assembles incomplete transport frames to preserve JSON and UTF-8. Each
frame and each redaction request (including both events) is limited to 2 MB;
oversized pairs fail with an SSE error. It never collects the full response.
Pairing adds a one-delta wait plus the regex HTTP call. `x-response-redaction-window: 2`
identifies this mode; the existing `buffered: false` headers mean no full-response
buffering, while one text delta can be waiting for its partner.

Final usage arrives in `response.completed`. This cell prints event arrival times and requires completion. Local tests hold back the terminal event to verify both deltas in a pair are delivered before completion.

In [ ]:
def run_stream(payload, expected_route, expected_search):
    started = time.perf_counter()
    first_event = None
    events = []
    with httpx.Client(timeout=240) as client:
        with client.stream('POST', gateway_url, headers=auth_headers(), json={**payload, 'stream': True}) as response:
            response.raise_for_status()
            assert response.headers['content-type'].startswith('text/event-stream')
            assert response.headers['x-lab-route'] == expected_route
            assert response.headers['x-lab-initial-buffered'] == 'false'
            if expected_route == 'stream-function':
                assert response.headers['x-lab-filter'] == 'conditional-regex'
                assert response.headers['x-response-redaction-window'] == '2'
                assert response.headers['x-response-metrics-status'] == 'streamed'
            for event in iter_sse(response.iter_lines()):
                assert event['type'] not in ('error', 'response.failed', 'response.incomplete')
                if first_event is None:
                    first_event = time.perf_counter() - started
                    print(f'First event after {first_event:.2f}s', flush=True)
                if event['type'].startswith('response.web_search_call.'):
                    print(f"[{time.perf_counter() - started:.2f}s] {event['type']}", flush=True)
                if event['type'] == 'response.output_text.delta':
                    print(event['delta'], end='', flush=True)
                events.append(event)
    completed = [event['response'] for event in events if event['type'] == 'response.completed']
    assert len(completed) == 1
    document = completed[0]
    assert invoked_web_search(document) == expected_search
    created = next((event['response']['id'] for event in events if event['type'] == 'response.created'), None)
    if created:
        assert document['id'] == created
    if expected_search:
        assert_redacted(document)  # The complete snapshot is independently filtered.
    print('\n', expected_route, len(events), 'events;', round(time.perf_counter() - started, 2), 'seconds')
    print('Original usage:', document.get('usage'), document.get('tool_usage'))
    return document

for payload, route, used in ((base_request, 'foundry', False),
                             (unused_request, 'stream-function', False),
                             (search_request, 'stream-function', True)):
    final_document = run_stream(payload, route, used)
    display(Markdown(output_text(final_document)))
print('Direct, unused-tool, and searched live SSE verified.')

## Troubleshooting and extensions
- **First event is delayed:** Check Function cold start, token acquisition, and Foundry latency. SSE is not buffered to completion. Text pairs add a one-delta wait and a regex HTTP call after search starts.
- **401/403:** Check the user audience/scope, APIM and Function Foundry roles, and Easy Auth permission for APIM plus the Function identity. Allow time for role propagation.
- **SSE error after HTTP 200:** The stream was malformed/truncated or a regex call failed. Already-delivered events cannot be recalled. Require `response.completed`.
- **No redaction despite an offered tool:** The relay invokes regex only after an actual search event. An unused tool remains unmodified.
- **Blocked URL visible across deltas:** Pairs are independent; URLs spanning separate pairs or more than two deltas can still be missed. Inspect the separately filtered final snapshot.
- **Plain names remain:** Domain names in prose are not URLs. The filter does not follow redirects or decode deliberate obfuscation.

As an extension, add a blocked domain, redeploy, and confirm blocked URLs split within a text-delta pair become `[BLOCKED LINK]`, including their paths and queries. Run [clean-up-resources.ipynb](clean-up-resources.ipynb) when finished.